In [1]:
import pandas as pd

df_insurance = pd.read_csv(
    "~/Downloads/auto_insurance_python.csv"
)

df_insurance.head()

,Customer ID,State,Coverage,Vehicle Class,Income,Months Since Policy Inception,Number of Policies,Customer Lifetime Revenue,Response
0,AA71604,Arizona,Extended,Two-Door Car,87560,86,2,8428,0.0
1,AB13432,California,Basic,SUV,0,16,1,1760,1.0
2,AB21519,California,Extended,Four-Door Car,93272,54,1,5130,0.0
3,AB31813,Washington,Extended,SUV,0,59,2,7729,0.0
4,AB60627,California,Extended,SUV,77517,20,2,2580,1.0


In [2]:
df_insurance = df_insurance.dropna(subset=["State"])

print(df_insurance.shape)

(1985, 9)


In [10]:
print(df_insurance["Income"].mean())

39005.91032745592


In [13]:
state_averages = df_insurance.groupby("State")[
    "Number of Policies"
].mean()

print(state_averages.loc[["Arizona", "Oregon"]])

State
Arizona    3.200514
Oregon     2.880200
Name: Number of Policies, dtype: float64


In [19]:
zero_income_count = (df_insurance["Income"] == 0).sum()

zero_income_percent = (
    zero_income_count / len(df_insurance)
) * 100

print("Customers with zero income:", zero_income_count)
print("Percentage:", round(zero_income_percent, 2))

Customers with zero income: 473
Percentage: 23.83


In [73]:
policy_counts = df_insurance["Number of Policies"].copy()

df_insurance.loc[
    policy_counts > 5, "Number of Policies"
] = 3

df_insurance.loc[
    (policy_counts > 1) & (policy_counts <= 5),
    "Number of Policies"
] = 2

df_insurance.loc[
    policy_counts == 1, "Number of Policies"
] = 1

In [4]:
df_insurance = df_insurance.drop(columns=["Customer ID"])

In [5]:
df_insurance = pd.get_dummies(
    df_insurance,
    columns=["State", "Coverage", "Vehicle Class"],
    drop_first=True,
    dtype=int
)

In [6]:
from sklearn.preprocessing import StandardScaler

scale_columns = [
    "Income",
    "Months Since Policy Inception",
    "Customer Lifetime Revenue"
]

scaler = StandardScaler()

scaler.fit(df_insurance[scale_columns])

df_insurance[scale_columns] = scaler.transform(
    df_insurance[scale_columns]
)

In [7]:
print(
    df_insurance["Number of Policies"]
    .value_counts()
    .sort_index()
)

Number of Policies
1    718
2    914
3    353
Name: count, dtype: int64


In [8]:
print(round(df_insurance["Income"].max(), 3))

1.995


In [9]:
print(df_insurance.shape[1])

16


In [10]:
df_prediction = df_insurance[
    df_insurance["Response"].isnull()
].copy()

In [11]:
df_insurance = df_insurance.dropna(subset=["Response"])

In [14]:
print("df_insurance:", len(df_insurance))
print("df_prediction:", len(df_prediction))

df_insurance: 1960
df_prediction: 25


In [15]:
X = df_insurance.drop(columns=["Response"])

y = df_insurance["Response"]

In [22]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=1
)

In [26]:
from sklearn.neural_network import MLPClassifier

modelNN = MLPClassifier(
    hidden_layer_sizes=(5,),
    random_state=1
)

In [29]:
modelNN.fit(X_train, y_train)

,hidden_layer_sizes,"(5,)"
,activation,'relu'
,solver,'adam'
,alpha,0.0001
,batch_size,'auto'
,learning_rate,'constant'
,learning_rate_init,0.001
,power_t,0.5
,max_iter,200
,shuffle,True
,random_state,1


In [33]:
predicted_test = modelNN.predict(X_test)

In [ ]:
Question 4

In [36]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score, recall_score

print("Accuracy:", accuracy_score(y_test, predicted_test))
print("Precision:", precision_score(y_test, predicted_test))
print("Recall:", recall_score(y_test, predicted_test))

Accuracy: 0.5136054421768708
Precision: 0.5142118863049095
Recall: 0.67003367003367


In [41]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, predicted_test))

[[103 188]
 [ 98 199]]


In [42]:
X_prediction = df_prediction.drop(columns=["Response"])

In [45]:
predicted_response = modelNN.predict(X_prediction)

In [50]:
print("Predicted to accept:", (predicted_response == 1).sum())
print("Predicted to reject:", (predicted_response == 0).sum())

Predicted to accept: 18
Predicted to reject: 7


In [56]:
accept_probabilities = modelNN.predict_proba(X_prediction)[:, 1]

print(accept_probabilities)

[0.43644217 0.65005893 0.56061994 0.48026028 0.50815244 0.47385821
 0.58653342 0.53738851 0.54650437 0.5826466  0.54765551 0.54266179
 0.5272202  0.63089688 0.624413   0.53008981 0.5725414  0.50252059
 0.61892837 0.58676864 0.4302007  0.43224674 0.47206164 0.55946746
 0.47877334]


In [66]:
predicted_accept_06 = accept_probabilities >= 0.6

In [71]:
print("Predicted to accept at 0.6:", predicted_accept_06.sum())

Predicted to accept at 0.6: 4
